# Analyse incrémentale exhaustive des familles de facteurs — EUROPE SMALL CAP

Ce notebook teste chaque variable des familles growth, quality, value, momentum et dividend avec le facteur historique de sa propre famille. Les tests size et low vol sont volontairement exclus.

Pour chaque famille et chaque dimension, chaque variable est ajoutée séparément avec une pondération **1:1** :
- facteur historique de la famille : `level=1.0` ;
- variable candidate : une seule dimension active à `1.0`.

Les dimensions testées sont le niveau, puis tous les changements `pct`, `diff` et `rank_diff` sur les horizons 1, 3, 6 et 12 mois. Les résultats incluent les deltas de rendement, d'IR, de pire CAGR, ainsi que deux scores relatifs de régularité et de persistance sur 0-100 ; aucune figure n'est produite.

Pour limiter la mémoire sous Windows, chaque lot ne transmet aux processus que les colonnes de sa famille ; les scores temporaires sont calculés dans les workers puis supprimés.


In [ ]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

from factor_config import FACTOR_FAMILIES, LOWER_IS_BETTER, signal_options
from func import (
    calculate_benchmark_performance,
    export_backtest_results,
    load_backtest_data,
    test_incremental_signals,
    _iter_backtest_results,
)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "func.py").exists():
    raise RuntimeError(
        "Le répertoire de travail Jupyter doit être la racine du projet."
    )

MARKET = "EUROPE SMALL CAP"
BENCHMARK = "MSCI EUR SMALL"
START_DATE = "2007-12-01"
PERCENTILE = 0.13
N_JOBS = 8
PERIOD_BREAKPOINTS = [2009, 2013, 2017, 2020, 2022, 2024, 2026]
OUTPUT_NAME = "factor_families_incremental_SMALL"
EXPORT_ROOT = REPO_ROOT / "exports"
LIST_NOIRE_PATH = None

CANDIDATE_DIMENSIONS = (
    "level",
    "pct_1", "pct_3", "pct_6", "pct_12",
    "diff_1", "diff_3", "diff_6", "diff_12",
    "rank_diff_1", "rank_diff_3", "rank_diff_6", "rank_diff_12",
)

BASELINE_CANDIDATES = {
    "growth": ("GROWTH_SCORE_FS_SECTOR", "Growth Avg Percentile"),
    "quality": ("Quality Avg Percentile", "MARGIN_SCORE_FS_SECTOR"),
    "value": ("VALUE_SCORE_FS_SECTOR", "Value Avg Percentile"),
    "momentum": ("MOMENTUM_SCORE_FS_SECTOR", "Mom Avg Percentile"),
    "dividend": ("Dividend Avg Percentile", "Dividend_NTM Avg Percentile"),
}
SELECTED_FAMILIES = ("growth", "quality", "value", "momentum", "dividend")

KEY_METRICS = (
    "active_cagr",
    "top_information_ratio",
    "top_worst_cagr",
    "robust_score",
    "active_max_drawdown",
    "tracking_error_annualized",
    "top_annualized_volatility",
    "top_max_drawdown",
)

available_columns = set(pq.ParquetFile(REPO_ROOT / "data" / "screen_aggregate.parquet").schema_arrow.names)
BASELINE_COLUMNS = {}
for family, candidates in BASELINE_CANDIDATES.items():
    selected = next((candidate for candidate in candidates if candidate in available_columns), None)
    if selected is None:
        raise KeyError(f"Aucun facteur historique disponible pour {family}: {candidates}")
    BASELINE_COLUMNS[family] = selected

missing_by_family = {
    family: [variable for variable in variables if variable not in available_columns]
    for family, variables in FACTOR_FAMILIES.items()
    if family in SELECTED_FAMILIES
}
missing_by_family = {
    family: missing for family, missing in missing_by_family.items() if missing
}
if missing_by_family:
    raise KeyError(f"Variables FACTOR_FAMILIES absentes du screen: {missing_by_family}")

def _safe_token(value):
    """Construit un identifiant portable pour les colonnes candidates."""
    return re.sub(r"[^A-Za-z0-9]+", "_", str(value)).strip("_") or "variable"

def _candidate_alias(family, dimension, variable):
    """Crée un alias distinct pour tester aussi les changements du facteur historique."""
    return f"__incremental__{family}__{dimension}__{_safe_token(variable)}"

def _baseline_config(family):
    """Construit le facteur historique de la famille avec un poids égal à un."""
    return {
        BASELINE_COLUMNS[family]: signal_options(
            level=1.0,
            higher_is_better=True,
        )
    }

print(f"Marché: {MARKET} | Benchmark: {BENCHMARK}")
print(f"Familles: {list(SELECTED_FAMILIES)}")
print(f"Dimensions candidates: {len(CANDIDATE_DIMENSIONS)}")
print(f"Baselines résolues: {BASELINE_COLUMNS}")


In [ ]:
DATA_DIR = REPO_ROOT / "data"
SCREEN_PATH = DATA_DIR / "screen_aggregate.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"

raw_variables = list(dict.fromkeys(
    variable
    for family in SELECTED_FAMILIES
    for variables in [FACTOR_FAMILIES[family]]
    for variable in variables
))
load_variables = list(dict.fromkeys(raw_variables + list(BASELINE_COLUMNS.values())))

screen, returns = load_backtest_data(
    screen_path=SCREEN_PATH,
    returns_path=RETURNS_PATH,
    variables=load_variables,
    bench=BENCHMARK,
    start_date=START_DATE,
    lookback_periods=12,
    compact_dtypes=True,
)
screen["Date"] = pd.to_datetime(screen["Date"])

missing_after_load = [
    variable for variable in load_variables if variable not in screen.columns
]
if missing_after_load:
    raise KeyError(f"Variables absentes après chargement: {missing_after_load}")
if f"Weight in {BENCHMARK}" not in screen.columns:
    raise KeyError(f"La colonne Weight in {BENCHMARK} est absente du screen.")

benchmark_performance = calculate_benchmark_performance(
    screen=screen,
    returns=returns,
    bench=BENCHMARK,
    start_date=START_DATE,
)
signal_columns = set(raw_variables) | set(BASELINE_COLUMNS.values())
structural_columns = [
    column for column in screen.columns if column not in signal_columns
]

RUN_OPTIONS = {
    "bench": BENCHMARK,
    "bench_perf": benchmark_performance,
    "percentile": PERCENTILE,
    "start_date": START_DATE,
    "freq_rebal": 1,
    "fill_method": "copy",
    "n_jobs": N_JOBS,
    "retain_builders": False,
    "monthly_base_cache": {},
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "show_plot": False,
    "build_figure": False,
}

incremental_batches = {"by_family": {}}
candidate_manifest_rows = []

for family in SELECTED_FAMILIES:
    variables = FACTOR_FAMILIES[family]
    family_variables = list(dict.fromkeys([
        *variables, BASELINE_COLUMNS[family],
    ]))
    family_screen = screen.loc[
        :, [*structural_columns, *family_variables]
    ].copy()
    incremental_batches["by_family"][family] = {}
    for dimension in CANDIDATE_DIMENSIONS:
        candidate_config = {}
        temporary_aliases = []
        for variable in variables:
            alias = variable
            if variable == BASELINE_COLUMNS[family]:
                alias = _candidate_alias(family, dimension, variable)
                family_screen[alias] = family_screen[variable]
                temporary_aliases.append(alias)
            candidate_config[alias] = signal_options(
                higher_is_better=variable not in LOWER_IS_BETTER,
                **{dimension: 1.0},
            )
            candidate_manifest_rows.append({
                "market": MARKET,
                "family": family,
                "baseline_variable": BASELINE_COLUMNS[family],
                "variable": variable,
                "dimension": dimension,
                "alias": alias,
                "higher_is_better": variable not in LOWER_IS_BETTER,
                "weight_baseline": 1.0,
                "weight_candidate": 1.0,
            })

        incremental_batches["by_family"][family][dimension] = test_incremental_signals(
            screen=family_screen,
            returns=returns,
            baseline_config=_baseline_config(family),
            candidate_config=candidate_config,
            list_noire_path=LIST_NOIRE_PATH,
            **RUN_OPTIONS,
        )
        incremental_batches["by_family"][family][dimension]["screen"] = None
        if temporary_aliases:
            family_screen.drop(columns=temporary_aliases, inplace=True)

candidate_manifest = pd.DataFrame(candidate_manifest_rows)
print(f"Observations screen: {screen.shape} | returns: {returns.shape}")
print(f"Lots incrémentaux: {len(SELECTED_FAMILIES) * len(CANDIDATE_DIMENSIONS)}")
print(f"Candidats variables × dimensions: {len(candidate_manifest)}")


In [ ]:
all_results = {"incremental": incremental_batches}
exported = export_backtest_results(
    results=all_results,
    output_dir=EXPORT_ROOT,
    export_name=OUTPUT_NAME,
    export_html=False,
    export_png=False,
    export_holdings=False,
)
EXPORT_DIR = Path(exported["export_dir"])
metrics = exported["metrics"].copy()
STABILITY_SCORE_COLUMNS = (
    "relative_smoothness_score",
    "relative_persistence_score",
)


def _relative_path_metrics(performance):
    """Mesure la régularité et la persistance de la courbe Top/Benchmark."""
    empty = {
        "path_trend_r2": np.nan,
        "path_residual_vol_annualized": np.nan,
        "relative_ulcer_index": np.nan,
        "median_recovery_months": np.nan,
        "positive_12m_active_rate": np.nan,
        "positive_36m_active_rate": np.nan,
        "monthly_active_hit_rate": np.nan,
    }
    if not isinstance(performance, pd.DataFrame):
        return empty
    required = {"Top", "Bench"}
    if not required.issubset(performance.columns):
        return empty
    frame = performance.loc[:, ["Top", "Bench"]].copy()
    frame.index = pd.to_datetime(frame.index, errors="coerce")
    frame = frame.loc[frame.index.notna()].sort_index()
    frame = frame.apply(pd.to_numeric, errors="coerce").dropna()
    if frame.empty:
        return empty
    monthly = frame.resample("ME").last().dropna()
    if len(monthly) < 2:
        return empty
    relative = (monthly["Top"] / monthly["Bench"]).replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
    if len(relative) < 2 or (relative <= 0).any():
        return empty

    log_relative = np.log(relative / relative.iloc[0])
    x_values = np.arange(len(log_relative), dtype=float)
    slope, intercept = np.polyfit(x_values, log_relative.to_numpy(), 1)
    fitted = slope * x_values + intercept
    residual = log_relative.to_numpy() - fitted
    total_sum_squares = float(
        np.square(log_relative.to_numpy() - log_relative.mean()).sum()
    )
    residual_sum_squares = float(np.square(residual).sum())
    trend_r2 = (
        1.0
        if total_sum_squares == 0
        else max(0.0, 1.0 - residual_sum_squares / total_sum_squares)
    )

    drawdown = relative / relative.cummax() - 1.0
    underwater = drawdown.lt(0)
    groups = underwater.ne(underwater.shift()).cumsum()
    recovery_durations = []
    for _, segment in underwater.groupby(groups):
        if not bool(segment.iloc[0]):
            continue
        if segment.index[-1] != underwater.index[-1]:
            recovery_durations.append(int(segment.sum()))

    active_returns = monthly["Top"].pct_change() - monthly["Bench"].pct_change()
    rolling_12m = relative.pct_change(12).dropna()
    rolling_36m = relative.pct_change(36).dropna()
    return {
        "path_trend_r2": trend_r2,
        "path_residual_vol_annualized": (
            float(np.std(residual, ddof=1) * np.sqrt(12))
            if len(residual) > 1
            else np.nan
        ),
        "relative_ulcer_index": float(np.sqrt(np.square(drawdown).mean())),
        "median_recovery_months": (
            float(np.median(recovery_durations))
            if recovery_durations
            else np.nan
        ),
        "positive_12m_active_rate": (
            float((rolling_12m > 0).mean()) if not rolling_12m.empty else np.nan
        ),
        "positive_36m_active_rate": (
            float((rolling_36m > 0).mean()) if not rolling_36m.empty else np.nan
        ),
        "monthly_active_hit_rate": (
            float((active_returns.dropna() > 0).mean())
            if active_returns.notna().any()
            else np.nan
        ),
    }


def _relative_stability_scores(path_metrics, strict_period_rate):
    """Réduit les diagnostics de trajectoire à deux scores sur 0-100."""
    def inverse(value, scale):
        if pd.isna(value):
            return np.nan
        return 1.0 / (1.0 + max(float(value), 0.0) / scale)

    smoothness_parts = [
        path_metrics.get("path_trend_r2"),
        inverse(path_metrics.get("path_residual_vol_annualized"), 0.15),
        inverse(path_metrics.get("relative_ulcer_index"), 0.10),
        (
            inverse(path_metrics.get("median_recovery_months"), 12.0)
            if pd.notna(path_metrics.get("median_recovery_months"))
            else 1.0
        ),
    ]
    persistence_parts = [
        path_metrics.get("positive_12m_active_rate"),
        path_metrics.get("positive_36m_active_rate"),
        path_metrics.get("monthly_active_hit_rate"),
        strict_period_rate,
    ]
    smoothness = pd.Series(smoothness_parts, dtype="float64").dropna()
    persistence = pd.Series(persistence_parts, dtype="float64").dropna()
    return {
        "relative_smoothness_score": (
            float(100.0 * smoothness.mean()) if not smoothness.empty else np.nan
        ),
        "relative_persistence_score": (
            float(100.0 * persistence.mean()) if not persistence.empty else np.nan
        ),
    }


def _stability_key(test_path, period_id, scope):
    """Construit une clé stable pour joindre les scores aux métriques."""
    return (str(test_path), str(period_id), str(scope))


performance_by_path = {
    test_path: result.get("performance")
    for test_path, result in _iter_backtest_results(all_results)
    if isinstance(result.get("performance"), pd.DataFrame)
}
period_definitions = metrics.loc[
    :,
    [
        "test_path",
        "scope",
        "period_id",
        "period_label",
        "actual_start_date",
        "actual_end_date",
    ],
].drop_duplicates(["test_path", "scope", "period_id"])

strict_rate_by_path = {}
for test_path, path_rows in metrics.groupby("test_path", dropna=True):
    subperiod_rows = path_rows.loc[path_rows["scope"].eq("subperiod")].dropna(
        subset=["active_cagr", "top_information_ratio", "top_worst_cagr"],
    )
    strict_mask = (
        subperiod_rows["active_cagr"].gt(0)
        & subperiod_rows["top_information_ratio"].gt(0)
        & subperiod_rows["top_worst_cagr"].gt(0)
    )
    strict_rate_by_path[str(test_path)] = (
        float(strict_mask.mean()) if not subperiod_rows.empty else np.nan
    )

stability_lookup = {}
for test_path, performance in performance_by_path.items():
    path_periods = period_definitions.loc[
        period_definitions["test_path"].eq(test_path)
    ]
    for _, period in path_periods.iterrows():
        period_id = period["period_id"]
        window = performance
        if str(period_id) != "total":
            start = pd.to_datetime(period["actual_start_date"], errors="coerce")
            end = pd.to_datetime(period["actual_end_date"], errors="coerce")
            if pd.notna(start) and pd.notna(end):
                indexed_performance = performance.copy()
                indexed_performance.index = pd.to_datetime(
                    indexed_performance.index, errors="coerce"
                )
                indexed_performance = indexed_performance.loc[
                    indexed_performance.index.notna()
                ].sort_index()
                window = indexed_performance.loc[start:end]
            else:
                window = performance.iloc[0:0]
        path_metrics = _relative_path_metrics(window)
        stability_lookup[
            _stability_key(test_path, period_id, period["scope"])
        ] = _relative_stability_scores(
            path_metrics,
            strict_rate_by_path.get(str(test_path), np.nan),
        )

candidate_lookup = candidate_manifest.set_index(
    ["family", "dimension", "alias"]
).to_dict(orient="index")
metric_columns = [column for column in KEY_METRICS if column in metrics.columns]
incremental_rows = []

for test_group, group_rows in metrics.groupby("test_group", dropna=True):
    group_text = str(test_group)
    group_parts = group_text.split(" / ")
    if len(group_parts) < 3 or group_parts[0] != "incremental":
        continue
    family, dimension = group_parts[-2], group_parts[-1]
    if family not in SELECTED_FAMILIES or dimension not in CANDIDATE_DIMENSIONS:
        continue

    baseline_rows = group_rows.loc[
        group_rows["test_type"].eq("incremental_baseline")
    ]
    candidate_rows = group_rows.loc[
        group_rows["test_type"].eq("incremental_candidate")
    ]
    for _, candidate in candidate_rows.iterrows():
        lookup = candidate_lookup.get((
            family, dimension, str(candidate["test_name"]),
        ))
        if lookup is None:
            raise KeyError(
                f"Alias candidat absent du manifest: {candidate['test_name']}"
            )
        baseline = baseline_rows.loc[
            (baseline_rows["period_id"].eq(candidate["period_id"]))
            & (baseline_rows["scope"].eq(candidate["scope"]))
        ]
        if baseline.empty:
            continue
        baseline = baseline.iloc[0]
        row = {
            **lookup,
            "period_id": candidate["period_id"],
            "scope": candidate["scope"],
            "period_label": candidate.get("period_label"),
            "candidate_test_path": candidate["test_path"],
            "baseline_test_path": baseline["test_path"],
        }
        for column in metric_columns:
            candidate_value = pd.to_numeric(candidate.get(column), errors="coerce")
            baseline_value = pd.to_numeric(baseline.get(column), errors="coerce")
            row[f"{column}_candidate"] = candidate_value
            row[f"{column}_baseline"] = baseline_value
            row[f"delta_{column}"] = candidate_value - baseline_value
        candidate_stability = stability_lookup.get(
            _stability_key(
                candidate["test_path"], candidate["period_id"], candidate["scope"]
            ),
            {},
        )
        baseline_stability = stability_lookup.get(
            _stability_key(
                baseline["test_path"], baseline["period_id"], baseline["scope"]
            ),
            {},
        )
        for score_column in STABILITY_SCORE_COLUMNS:
            row[f"{score_column}_candidate"] = candidate_stability.get(
                score_column, np.nan
            )
            row[f"{score_column}_baseline"] = baseline_stability.get(
                score_column, np.nan
            )
            row[f"delta_{score_column}"] = (
                row[f"{score_column}_candidate"]
                - row[f"{score_column}_baseline"]
            )

        row["incremental_perf_improved"] = (
            row["delta_active_cagr"] > 0
            and row["delta_top_information_ratio"] > 0
            and row["delta_top_worst_cagr"] > 0
        )
        row["incremental_risk_not_worse"] = (
            row["delta_active_max_drawdown"] <= 0
            and row["delta_tracking_error_annualized"] <= 0
        )
        row["incremental_absolute_risk_not_worse"] = (
            row.get("delta_top_annualized_volatility", np.nan) <= 0
            and row.get("delta_top_max_drawdown", np.nan) <= 0
        )
        incremental_rows.append(row)

incremental_effects = pd.DataFrame(incremental_rows)
if incremental_effects.empty:
    raise RuntimeError("Aucune ligne incrémentale n'a été produite.")

incremental_effects_all_periods = incremental_effects.sort_values(
    ["family", "variable", "dimension", "scope", "period_id"]
).reset_index(drop=True)
incremental_effects_total = incremental_effects.loc[
    incremental_effects["period_id"].eq("total")
].sort_values(
    ["family", "delta_active_cagr"],
    ascending=[True, False],
).reset_index(drop=True)

subperiods = incremental_effects.loc[
    incremental_effects["scope"].eq("subperiod")
].copy()
if not subperiods.empty:
    consistency = subperiods.groupby(
        ["family", "baseline_variable", "variable", "dimension", "higher_is_better"],
        dropna=False,
    ).agg(
        subperiod_count=("period_id", "count"),
        positive_active_cagr_rate=("delta_active_cagr", lambda values: float((values > 0).mean())),
        positive_information_ratio_rate=("delta_top_information_ratio", lambda values: float((values > 0).mean())),
        positive_worst_cagr_rate=("delta_top_worst_cagr", lambda values: float((values > 0).mean())),
        performance_improved_rate=("incremental_perf_improved", "mean"),
        positive_smoothness_rate=("delta_relative_smoothness_score", lambda values: float((values.dropna() > 0).mean()) if values.notna().any() else np.nan),
        positive_persistence_rate=("delta_relative_persistence_score", lambda values: float((values.dropna() > 0).mean()) if values.notna().any() else np.nan),
        risk_not_worse_rate=("incremental_risk_not_worse", "mean"),
        absolute_risk_not_worse_rate=("incremental_absolute_risk_not_worse", "mean"),
    ).reset_index()
    strict_mask = (
        subperiods["incremental_perf_improved"]
        & subperiods["incremental_risk_not_worse"]
        & subperiods["incremental_absolute_risk_not_worse"]
    )
    strict_rates = subperiods.assign(strict_improvement=strict_mask).groupby(
        ["family", "baseline_variable", "variable", "dimension"],
        dropna=False,
    )["strict_improvement"].mean().rename("strict_improvement_rate")
    consistency = consistency.merge(
        strict_rates.reset_index(),
        on=["family", "baseline_variable", "variable", "dimension"],
        how="left",
    )
else:
    consistency = pd.DataFrame()

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
candidate_manifest.to_csv(EXPORT_DIR / "candidate_manifest.csv", index=False)
incremental_effects_all_periods.to_csv(
    EXPORT_DIR / "incremental_effects_all_periods.csv",
    index=False,
)
incremental_effects_total.to_csv(
    EXPORT_DIR / "incremental_effects_total.csv",
    index=False,
)
consistency.to_csv(EXPORT_DIR / "incremental_consistency.csv", index=False)

run_manifest = {
    "market": MARKET,
    "benchmark": BENCHMARK,
    "percentile": PERCENTILE,
    "n_jobs": N_JOBS,
    "weighting": "baseline level 1.0 + one candidate dimension 1.0",
    "candidate_dimensions": list(CANDIDATE_DIMENSIONS),
    "families": list(SELECTED_FAMILIES),
    "baseline_columns": BASELINE_COLUMNS,
    "outputs": [
        "backtest_metrics.csv",
        "backtest_registry.json",
        "candidate_manifest.csv",
        "incremental_effects_all_periods.csv",
        "incremental_effects_total.csv",
        "incremental_consistency.csv",
    ],
}
with (EXPORT_DIR / "run_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(run_manifest, handle, ensure_ascii=False, indent=2)

total_display_columns = [
    "family",
    "baseline_variable",
    "variable",
    "dimension",
    "delta_active_cagr",
    "delta_top_information_ratio",
    "delta_top_worst_cagr",
    "delta_relative_smoothness_score",
    "delta_relative_persistence_score",
    "delta_robust_score",
    "delta_active_max_drawdown",
    "delta_tracking_error_annualized",
    "incremental_perf_improved",
    "incremental_risk_not_worse",
    "incremental_absolute_risk_not_worse",
]
consistency_display_columns = [
    "family",
    "baseline_variable",
    "variable",
    "dimension",
    "subperiod_count",
    "positive_active_cagr_rate",
    "positive_information_ratio_rate",
    "positive_worst_cagr_rate",
    "positive_smoothness_rate",
    "positive_persistence_rate",
    "performance_improved_rate",
    "risk_not_worse_rate",
    "absolute_risk_not_worse_rate",
    "strict_improvement_rate",
]

print(f"Répertoire des résultats: {EXPORT_DIR}")
print("Tableau total: chaque ligne est une variable × dimension.")
display(incremental_effects_total.loc[
    :, [column for column in total_display_columns if column in incremental_effects_total.columns]
])
print("Tableau de persistance: taux de périodes positives.")
display(consistency.loc[
    :, [column for column in consistency_display_columns if column in consistency.columns]
])
